# MAE1(SpatialODMAE1) Loss 탐색 — Colab GPU 전용 노트북

- 브랜치: `experiment/loss-comparison`
- 이 노트북은 loss 실험 전용이며, 팀 공용 `src/colab.ipynb`(mae1/twostage 기존 baseline 셀)와는 무관하다.
- **소스 코드는 `/content/KTxTS_project`(Colab VM 로컬 디스크)에서 실행**하고(git 연산이 Drive 마운트보다 훨씬 빠름), **체크포인트/CSV/로그/리포트만 Google Drive 절대경로**(`/content/drive/MyDrive/kt/loss_experiments/<RUN_ID>`)에 저장한다.
- 세션이 끊겨도 같은 RUN_ID로 이 노트북을 다시 열고 0번 셀부터 재실행하면, train.py의 full training state resume(모델/옵티마이저/스케줄러/epoch/RNG)과 오케스트레이터의 skip-if-done(completed.json/training_state.pt 기준) 덕분에 진행 중이던 지점부터 이어서 실행된다.

## 0-1. Google Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0-2. 저장소 clone 또는 fetch (`/content`, 로컬 디스크)
이미 `/content/KTxTS_project`가 있으면(같은 런타임에서 재실행) fetch+checkout만 하고, 없으면 새로 clone한다.

In [ ]:
import os

REPO_DIR = '/content/KTxTS_project'
REPO_URL = 'https://github.com/implement-k/KTxTS_project'
BRANCH = 'experiment/loss-comparison'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

%cd $REPO_DIR
!git fetch origin
!git checkout $BRANCH
!git pull origin $BRANCH

## 0-3. 브랜치 이름과 commit hash 출력 (실행 조건 기록용)

In [ ]:
!git branch --show-current
!git rev-parse HEAD
!git status --short

## 0-4. 패키지 확인 및 설치
Colab 기본 이미지에 보통 torch/numpy/pandas/scikit-learn/matplotlib은 이미 있다. 이 저장소에서 추가로 필요한 것만 설치한다.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q lightgbm joblib

## 0-5. CUDA GPU 확인
GPU가 안 잡히면(False) Colab 런타임 유형을 GPU로 바꾼 뒤(런타임 > 런타임 유형 변경) 다시 실행할 것.

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('CUDA GPU를 사용할 수 없습니다 — 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.')

## 0-6. RUN_ID / 결과 저장용 Drive 절대경로 설정

**새로 시작**: 아래 셀을 그대로 실행하면 새 타임스탬프 RUN_ID가 생성된다.

**이어서 하기(세션이 끊겼다 재연결된 경우)**: 자동 생성 줄을 주석 처리하고, 이전에 출력됐던 RUN_ID를 직접 지정한 뒤 실행할 것.

In [ ]:
import time

# 새로 시작: 아래 줄 사용
RUN_ID = time.strftime('%Y%m%d_%H%M%S')
# 이어서 하기: 위 줄을 주석 처리하고 아래처럼 이전 RUN_ID를 직접 지정
# RUN_ID = '20260716_090000'

OUT_DIR = f'/content/drive/MyDrive/kt/loss_experiments/{RUN_ID}'
import os
os.makedirs(OUT_DIR, exist_ok=True)
os.environ['OUT_DIR'] = OUT_DIR
os.environ['PY_EXE'] = os.popen('which python3').read().strip()
print('RUN_ID =', RUN_ID)
print('OUT_DIR =', OUT_DIR, '(Drive 절대경로 — 세션이 끊겨도 보존됨)')
print('PY_EXE =', os.environ['PY_EXE'])

---
## 1. Target 분포 분석 (strict protocol)
신규 loss(dual_scale_mse, bin_balanced_mse)의 데이터 기반 하이퍼파라미터를 산출한다. Stage 1 이전에 반드시 1회 필요. `derived_loss_hparams_strict.json`이 생성되어야 이후 Stage가 동작한다.

In [ ]:
!python scripts/analyze_target_distribution.py --out-dir "$OUT_DIR" --protocol strict --n-mask-samples 300

## 2. Stage 0 — Historical baseline reproduction (legacy protocol)
CPC 0.5434 / RMSE 924.49(노션 기록)와 비교한다. legacy protocol은 test 지역이 checkpoint 선택에도 쓰이는 재현 전용 프로토콜이며, 이후 신규 loss 순위 선정에는 쓰이지 않는다.

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$OUT_DIR" --python "$PY_EXE" --stage 0

## 3. Stage 1 — Runtime smoke (strict protocol, 후보당 30 step)
순위에는 사용하지 않음. forward/backward, NaN/inf, gradient norm/clip 비율, loss 내부 term, checkpoint/로그 생성 여부만 확인.

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$OUT_DIR" --python "$PY_EXE" --stage 1

## 4. Stage 2 — Screening (strict protocol, planned 50 epoch 중 15 epoch까지)
baseline + Stage 1 통과 후보를 대상으로 strict validation(strict_val_indices) 성능을 기록한다.

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$OUT_DIR" --python "$PY_EXE" --stage 2

## 5. Stage 3 — Final (strict protocol, 50 epoch, 최종 test는 여기서만 실행)
baseline + screening 통과 상위 후보(최대 2개)를 50 epoch까지 이어서 학습하고, 이때만 test_indices 최종 평가를 수행한다. 예상 시간이 --time-budget-hours를 넘으면 후보를 자동 축소한다.

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$OUT_DIR" --python "$PY_EXE" --stage 3 --time-budget-hours 8

## 6. 최종 리포트 확인
`final_report.md`를 생성/갱신한다(어느 시점에 다시 실행해도 디스크에 남은 결과로부터 재구성되어 안전).

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$OUT_DIR" --python "$PY_EXE" --stage report

with open(f'{OUT_DIR}/final_report.md', encoding='utf-8') as f:
    print(f.read())

## 결과 확인 경로 (모두 Drive 절대경로, 세션 종료 후에도 보존)
```
$OUT_DIR/RUN_STATUS.md
$OUT_DIR/target_distribution_strict.{csv,md}
$OUT_DIR/derived_loss_hparams_strict.json
$OUT_DIR/stage0_historical_reproduction.csv
$OUT_DIR/screening_leaderboard.csv
$OUT_DIR/final_leaderboard.csv
$OUT_DIR/final_report.md
$OUT_DIR/runs/<run>/attempt_NN/train.log, training_state.pt, completed.json, best_model_*.pth
```
세션이 끊겼다면: 이 노트북을 다시 열고 0-6번 셀에서 RUN_ID를 위와 동일하게 지정한 뒤, 0-1~0-5번(mount/clone/설치/CUDA 확인)부터 다시 실행하고 이후 셀은 순서대로 다시 실행하면 된다 — 이미 끝난 조합/epoch는 자동으로 건너뛰고 이어서 진행된다.